# Algoritmo de Rabin-Karp

Busca un patrón de longitud $m$ comparando hashes en lugar de cadenas. Se usa un hash polinómico con base $B$ y módulo $M$:

$$h(s_0 \dots s_{m-1}) = \sum_{k=0}^{m-1} s_k B^{m-1-k} \bmod M$$

que es rodante: el de la ventana siguiente sale del anterior en $O(1)$ quitando el carácter que sale y añadiendo el que entra,

$$h_{i+1} = \left((h_i - s_i B^{m-1}) B + s_{i+m}\right) \bmod M$$

Solo cuando los hashes coinciden se comparan los caracteres, para descartar colisiones. Cuesta $O(n + m)$ de media y $O(nm)$ en el peor caso (si hay muchas colisiones).

## 1. Hash rodante

Calcular el hash de todas las subcadenas de longitud $m$ actualizando el anterior en $O(1)$ y comprobar que coincide con calcularlo desde cero.

In [1]:
import random
import timeit

BASE, MOD = 256, 1_000_000_007

def hash_cadena(s, base=BASE, mod=MOD):
    h = 0
    for c in s:
        h = (h * base + ord(c)) % mod
    return h

def hashes_ventana(texto, m, base=BASE, mod=MOD):
    if m > len(texto):
        return []
    potencia = pow(base, m - 1, mod)
    h = hash_cadena(texto[:m], base, mod)
    hashes = [h]
    for i in range(len(texto) - m):
        h = ((h - ord(texto[i]) * potencia) * base + ord(texto[i + m])) % mod
        hashes.append(h)
    return hashes

texto = "abracadabra"
for i, h in enumerate(hashes_ventana(texto, 4)):
    print(f"{texto[i:i + 4]}  {h:>10}")

random.seed(1)
for _ in range(300):
    t = "".join(random.choices("abc", k=random.randint(0, 20)))
    m = random.randint(1, 6)
    assert hashes_ventana(t, m) == [hash_cadena(t[i:i + m]) for i in range(len(t) - m + 1)]

abra   633841754
brac   651663196
raca   918985050
acad   633902941
cada   667327066
adab   633968475
dabr   684103787
abra   633841754


## 2. Búsqueda de patrón

Buscar todas las apariciones de un patrón comparando hashes y, solo cuando coinciden, verificando los caracteres. Comprobar contra la búsqueda ingenua.

In [2]:
def rabin_karp(texto, patron, mod=MOD):
    m, objetivo = len(patron), hash_cadena(patron, BASE, mod)
    return [i for i, h in enumerate(hashes_ventana(texto, m, BASE, mod))
            if h == objetivo and texto[i:i + m] == patron]

def ingenua(texto, patron):
    m = len(patron)
    return [i for i in range(len(texto) - m + 1) if texto[i:i + m] == patron]

for _ in range(500):
    t = "".join(random.choices("ab", k=30))
    p = "".join(random.choices("ab", k=random.randint(1, 4)))
    assert rabin_karp(t, p) == ingenua(t, p)
print(rabin_karp("abracadabra", "abra"), rabin_karp("AAAAA", "AA"))

[0, 7] [0, 1, 2, 3]


## 3. Colisiones

Con un módulo pequeño, muchas ventanas distintas del patrón comparten su hash (falsos positivos) y hay que verificarlas una a una. Contar las colisiones en un texto aleatorio para varios módulos (se esperan unas $n / M$), comprobar que el resultado de la búsqueda no cambia y medir cuánto cuesta verificar.

In [3]:
def colisiones(texto, patron, mod):
    m, objetivo = len(patron), hash_cadena(patron, BASE, mod)
    return sum(h == objetivo and texto[i:i + m] != patron
               for i, h in enumerate(hashes_ventana(texto, m, BASE, mod)))

random.seed(3)
texto = "".join(random.choices("abcdefghij", k=100_000))
patron = "abcde"
esperado = ingenua(texto, patron)
for mod in (7, 101, 10_007, 1_000_000_007):
    assert rabin_karp(texto, patron, mod) == esperado
    tiempo = timeit.timeit(lambda: rabin_karp(texto, patron, mod), number=1)
    print(f"M = {mod:>10}: {colisiones(texto, patron, mod):6} colisiones, "
          f"{len(esperado)} apariciones, {tiempo:.3f} s")

M =          7:  14343 colisiones, 3 apariciones, 0.014 s
M =        101:    999 colisiones, 3 apariciones, 0.013 s
M =      10007:     11 colisiones, 3 apariciones, 0.015 s
M = 1000000007:      0 colisiones, 3 apariciones, 0.020 s


## 4. Subcadena repetida más larga

Encontrar la subcadena más larga que aparece al menos dos veces (pueden solaparse). Si hay una repetida de longitud $L$, también la hay de cualquier longitud menor, así que $L$ se busca con búsqueda binaria; para cada longitud candidata, los hashes rodantes detectan una repetición en $O(n)$. En total, $O(n \log n)$. Comprobar contra la fuerza bruta y aplicarlo a una cadena de ADN aleatoria.

In [4]:
def repetida_de_longitud(texto, m):
    vistos = {}
    for i, h in enumerate(hashes_ventana(texto, m)):
        if any(texto[j:j + m] == texto[i:i + m] for j in vistos.get(h, [])):
            return texto[i:i + m]
        vistos.setdefault(h, []).append(i)
    return None

def repetida_mas_larga(texto):
    izq, der, mejor = 1, len(texto) - 1, ""
    while izq <= der:
        medio = (izq + der) // 2
        encontrada = repetida_de_longitud(texto, medio)
        if encontrada:
            mejor, izq = encontrada, medio + 1
        else:
            der = medio - 1
    return mejor

def repetida_bruta(texto):
    n = len(texto)
    return max((texto[i:j] for i in range(n) for j in range(i + 1, n + 1) if texto.find(texto[i:j], i + 1) != -1),
               key=len, default="")

for _ in range(500):
    t = "".join(random.choices("ab", k=random.randint(0, 14)))
    assert len(repetida_mas_larga(t)) == len(repetida_bruta(t))
print(repetida_mas_larga("banana"), repetida_mas_larga("abracadabra"))

random.seed(4)
adn = "".join(random.choices("ACGT", k=20_000))
repetida = repetida_mas_larga(adn)
print(len(repetida), repetida, ingenua(adn, repetida))

ana abra
13 CCACCTTCATTAG [14601, 17331]


## 5. Varios patrones a la vez

Buscar a la vez varios patrones de la misma longitud guardando sus hashes en un diccionario: un solo recorrido del texto en lugar de uno por patrón. Comprobar contra la búsqueda ingenua.

In [5]:
def rabin_karp_varios(texto, patrones):
    m = len(patrones[0])
    por_hash = {}
    for p in patrones:
        por_hash.setdefault(hash_cadena(p), []).append(p)
    encontrados = {p: [] for p in patrones}
    for i, h in enumerate(hashes_ventana(texto, m)):
        for p in por_hash.get(h, []):
            if texto[i:i + m] == p:
                encontrados[p].append(i)
    return encontrados

random.seed(5)
for _ in range(300):
    t = "".join(random.choices("abc", k=40))
    m = random.randint(1, 4)
    patrones = sorted({"".join(random.choices("abc", k=m)) for _ in range(4)})
    assert rabin_karp_varios(t, patrones) == {p: ingenua(t, p) for p in patrones}
print(rabin_karp_varios("el gato y el pato van al plato", ["ato", "van", "el "]))

{'ato': [4, 14, 27], 'van': [18], 'el ': [0, 10]}
